This was ported over from the TabM example with Gemini.

In [1]:
!pip install delu rtdl_revisiting_models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing in

In [2]:
import pandas as pd
import numpy as np
import math

from tqdm import tqdm

import sklearn
import sklearn.model_selection

import torch
import torch.nn.functional as F

from rtdl_revisiting_models import FTTransformer

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [3]:
try:
    df = pd.read_csv('sparcs.csv', low_memory=False)
except:
    df = pd.read_csv('/kaggle/input/sparcs/sparcs.csv', low_memory=False)

In [4]:
df.head()

,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,50 to 69,107,F,White,Not Span/Hispanic,...,Major,Major,Medical,Medicaid,NaN,NaN,NaN,Y,"51,514.62","7,552.54"
1,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,M,Black/African American,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"25,370.86","3,469.55"
2,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,104,F,Other Race,Spanish/Hispanic,...,Minor,Minor,Medical,Medicaid,NaN,NaN,NaN,N,"23,876.78","6,180.33"
3,New York City,Bronx,7000006.0,3058.0,Montefiore Med Center - Jack D Weiler Hosp of ...,18 to 29,100,F,Black/African American,Not Span/Hispanic,...,Moderate,Minor,Medical,Medicaid,NaN,NaN,NaN,Y,"43,319.05","12,588.93"
4,New York City,Bronx,7000006.0,1168.0,Montefiore Medical Center-Wakefield Hospital,18 to 29,104,M,Other Race,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,NaN,NaN,NaN,Y,"40,266.23","10,355.99"


In [5]:
df.shape

(2103433, 33)

In [6]:
summary = pd.DataFrame({
    'nunique': df.nunique(),
    'dtype': df.dtypes
})

print(summary)

                                     nunique    dtype
Hospital Service Area                      8   object
Hospital County                           57   object
Operating Certificate Number             167  float64
Permanent Facility Id                    206  float64
Facility Name                            206   object
Age Group                                  5   object
Zip Code - 3 digits                       50   object
Gender                                     3   object
Race                                       4   object
Ethnicity                                  4   object
Length of Stay                           120   object
Type of Admission                          6   object
Patient Disposition                       19   object
Discharge Year                             1    int64
CCSR Diagnosis Code                      480   object
CCSR Diagnosis Description               480   object
CCSR Procedure Code                      321   object
CCSR Procedure Description  

In [7]:
df[['APR DRG Code', 'APR DRG Description']].drop_duplicates()

,APR DRG Code,APR DRG Description
0,137,MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS
1,383,CELLULITIS AND OTHER SKIN INFECTIONS
2,560,VAGINAL DELIVERY
4,750,SCHIZOPHRENIA
5,138,BRONCHIOLITIS AND RSV PNEUMONIA
...,...,...
48263,692,RADIOTHERAPY
50812,603,NEONATE BIRTH WEIGHT 1000-1249 GRAMS WITH OR W...
51078,177,CARDIAC PACEMAKER AND DEFIBRILLATOR REVISION E...
101808,863,NEONATAL AFTERCARE


Hospital Service Area                      8   object Categorical
Hospital County                           57   object Categorical
Operating Certificate Number             167  float64 Categorical
Permanent Facility Id                    206  float64 Categorical
Age Group                                  5   object Categorical
Zip Code - 3 digits                       50   object Categorical
Gender                                     3   object Categorical
Race                                       4   object Categorical
Ethnicity                                  4   object Categorical
Length of Stay                           120   object Integer
Type of Admission                          6   object Categorical
Patient Disposition                       19   object Categorical
Discharge Year                             1    int64 Integer
CCSR Diagnosis Code                      480   object Categorical
CCSR Procedure Code                      321   object Categorical
APR DRG Code                             334    int64 Categorical
APR MDC Code                              26    int64 Categorical
APR Severity of Illness Code               5    int64 Categorical
APR Risk of Mortality                      4   object Categorical
APR Medical Surgical Description           3   object Categorical
Payment Typology 1                         9   object Categorical
Payment Typology 2                         9   object Categorical
Payment Typology 3                         9   object Categorical
Birth Weight                              73   object Integer
Emergency Department Indicator             2   object Categorical
Total Charges                        1837017   object Integer
Total Costs                          1579320   object Integer


Facility Name                            206   object
CCSR Diagnosis Description               480   object
CCSR Procedure Description               321   object
APR DRG Description                      334   object
APR Severity of Illness Description        4   object
APR MDC Description                       26   object



In [8]:
df_cleaned = df

# Remove redundant columns, this information is contained in other columns
# E.g. CCSR Diagnosis Code and CCSR Diagnosis Description
df_cleaned = df_cleaned.drop(['Facility Name',
                      'CCSR Diagnosis Description',
                      'CCSR Procedure Description',
                      'APR DRG Description',
                      'APR Severity of Illness Description',
                      'APR MDC Description'
                      ], axis=1)

# Convert the Total Charges/Costs columns to numeric
# Remove commas that Pandas can not parse
df_cleaned['Total Charges'] = pd.to_numeric(df_cleaned['Total Charges'].str.replace(',', '', regex=False))
df_cleaned['Total Costs'] = pd.to_numeric(df_cleaned['Total Costs'].str.replace(',', '', regex=False))

# Convert the Length of Stay column to Numeric
# Right now we just map 120+ days to 120
df_cleaned['Length of Stay'] = pd.to_numeric(df_cleaned['Length of Stay'].replace({'120 +': '120'}))


# TODO: Check how NaNs and other weird stuff are dealt with
categorical_columns = ['Hospital Service Area', 'Hospital County', 'Operating Certificate Number', 'Permanent Facility Id', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'Patient Disposition', 'CCSR Diagnosis Code', 'CCSR Procedure Code', 'APR DRG Code', 'APR MDC Code', 'APR Severity of Illness Code', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Emergency Department Indicator']

df_cleaned[categorical_columns] = df_cleaned[categorical_columns].astype('category')


# It seems like there are 1.8 million NAs for birthweight, likely adults
# Also check if UNKN should be separate or something
# We can cast to Int64 because the normal int is not nullable,
# but instead we will just map nans to 0 for now, because I'm
# not sure how the neural network will deal with nan values
df_cleaned['Birth Weight'] = pd.to_numeric(df_cleaned['Birth Weight'].replace({'UNKN': 0, np.nan: 0}))


# Dropping the Total Charges column for now. We need to figure out
# what it actually means and if using it is possible at inference
# time
# df_cleaned = df_cleaned.drop(['Total Charges'], axis=1)

In [9]:
summary = pd.DataFrame({
    'nunique': df_cleaned.nunique(),
    'dtype': df_cleaned.dtypes
})

print(summary)

                                  nunique     dtype
Hospital Service Area                   8  category
Hospital County                        57  category
Operating Certificate Number          167  category
Permanent Facility Id                 206  category
Age Group                               5  category
Zip Code - 3 digits                    50  category
Gender                                  3  category
Race                                    4  category
Ethnicity                               4  category
Length of Stay                        120     int64
Type of Admission                       6  category
Patient Disposition                    19  category
Discharge Year                          1     int64
CCSR Diagnosis Code                   480  category
CCSR Procedure Code                   321  category
APR DRG Code                          334  category
APR MDC Code                           26  category
APR Severity of Illness Code            5  category
APR Risk of 

In [10]:
cont_cols = df_cleaned.select_dtypes(exclude=['category']).columns.drop('Total Costs')
X_cont = df_cleaned[cont_cols].to_numpy(dtype='float32')
n_cont_features = X_cont.shape[1]

cat_cols = df_cleaned.select_dtypes(include=['category']).columns
X_cat = df_cleaned[cat_cols].apply(lambda s: s.cat.codes).to_numpy(dtype='int64')

per_col_max = X_cat.max(axis=0)
cat_cardinalities = (per_col_max + 2).tolist()
print(f"Categorical Variable Cardinalities: {cat_cardinalities}")

# TODO: Temporarily?
replace_nans_with = per_col_max + 1
X_cat = np.where(X_cat == -1, replace_nans_with[np.newaxis, :], X_cat)


y = df_cleaned['Total Costs'].to_numpy()

indices = list(range(len(y)))

train_indices, test_indices = sklearn.model_selection.train_test_split(indices, test_size=0.2, random_state=50)
train_indices, val_indices = sklearn.model_selection.train_test_split(train_indices, test_size=0.2, random_state=50)

data = {
    'train': {'x_cont': X_cont[train_indices], 'x_cat': X_cat[train_indices], 'y': y[train_indices]},
    'val': {'x_cont': X_cont[val_indices], 'x_cat': X_cat[val_indices], 'y': y[val_indices]},
    'test': {'x_cont': X_cont[test_indices], 'x_cat': X_cat[test_indices], 'y': y[test_indices]},
}

Categorical Variable Cardinalities: [9, 58, 168, 207, 6, 51, 4, 5, 5, 7, 20, 481, 322, 335, 27, 6, 5, 4, 10, 10, 10, 3]


**The code after this is taken from the official TabM example.**

In [ ]:
X_cont_train_numpy = data['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_indices) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)

del X_cont_train_numpy


for part in data:
    data[part]['x_cont'] = preprocessing.transform(data[part]['x_cont'])

Y_train = data['train']['y'].copy()
y_mean = Y_train.mean().item()
y_std = Y_train.std().item()
Y_train = (Y_train - y_mean) / y_std

In [ ]:
# From C
import joblib
import json

joblib.dump(preprocessing, 'quantile_transformer.joblib')

with open('target_scaler.json', 'w') as f:
    json.dump({'mean': y_mean, 'std': y_std}, f)


In [ ]:
model = FTTransformer(
    n_cont_features=n_cont_features,
    cat_cardinalities=cat_cardinalities,
    d_out=1,
    **FTTransformer.get_default_kwargs(),
).to(device)
optimizer = model.make_default_optimizer()

compile_model = False

if compile_model:
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data[part].items()}
    for part in data
}
Y_train = torch.as_tensor(Y_train, device=device)

for part in data:
    data[part]['y'] = data[part]['y'].float()
Y_train = Y_train.float()

amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None

compile_model = False

print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)

In [ ]:
@torch.autocast(torch.device(device).type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: torch.Tensor) -> torch.Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx],
        )
        .squeeze(-1)
        .float()
    )

loss_fn = F.mse_loss

@evaluation_mode()
def evaluate(part: str) -> float:
    model.eval()

    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    y_pred = y_pred * y_std + y_mean

    y_true = data[part]['y'].cpu().numpy()

    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
    )
    rmse = -score
    mae  = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    r2   = sklearn.metrics.r2_score(y_true, y_pred)

    print(f"[{part}] RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

    return float(score)  # The higher -- the better.

print(f'Test score before training: {evaluate("test"):.4f}')

In [ ]:
import math
import os
from tqdm import tqdm

ckpt_dir = '/kaggle/working'
os.makedirs(ckpt_dir, exist_ok=True)

n_epochs = 200
patience = 8

train_size = len(train_indices)
batch_size = 256
epoch_size = math.ceil(train_size / batch_size)
best = {
    'val': -math.inf,
    'test': -math.inf,
    'epoch': -1,
}

# patience = 16
remaining_patience = patience

print('-' * 88 + '\n')
for epoch in range(n_epochs):
    batches = torch.randperm(train_size, device=device).split(batch_size)
    for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
        if grad_scaler is None:
            loss.backward()
            optimizer.step()
        else:
            grad_scaler.scale(loss).backward()  # type: ignore
            grad_scaler.step(optimizer)
            grad_scaler.update()

    val_score = evaluate('val')
    test_score = evaluate('test')
    print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

    if val_score > best['val']:
        print('🌸 New best epoch! 🌸')
        best = {'val': val_score, 'test': test_score, 'epoch': epoch}
        remaining_patience = patience

        ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')
        torch.save(model.state_dict(), ckpt_path)
    else:
        remaining_patience -= 1

    if remaining_patience < 0:
        break

    print()

print('\n\nResult:')
print(best)

In [ ]:
state_dict = torch.load('/kaggle/working/best_model.pt', map_location=device)
model.load_state_dict(state_dict)

In [ ]:
print(f'Test score after training: {evaluate("test"):.4f}')